In [ ]:
!pip install pytesseract

In [ ]:
import cv2
import pytesseract
from PIL import Image
import numpy as np
import os
import re
from scipy.stats import norm
from scipy.optimize import brentq, least_squares
from datetime import datetime, timedelta
import math
import pandas as pd

In [ ]:
# --- Black-Scholes price function ---
def bs_price(S, K, T, r, sigma, is_call=True):
    """Compute Black-Scholes option price."""
    if sigma <= 0:
        return 0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if is_call:
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

# --- Implied volatility solver ---
def implied_vol(S, K, T, r, price, is_call=True):
    """Compute implied volatility using Brent's method."""
    try:
        f = lambda sigma: bs_price(S, K, T, r, sigma, is_call) - price
        return brentq(f, 1e-6, 5.0, maxiter=500)  # volatility between X and X%
    except ValueError:
        return np.nan  # No valid root found

def bs_greeks(S, K, T, r=0.05, q=0.0, sigma=0.2, is_call=True):
    """
    Compute Black–Scholes Greeks for calls and puts.
    """

    if T <= 0 or sigma <= 0:
        return {g: np.nan for g in ["delta", "gamma", "vega", "theta", "rho"]}

    sqrtT = math.sqrt(T)
    d1 = (math.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT

    pdf_d1 = norm.pdf(d1)
    cdf_d1 = norm.cdf(d1)
    cdf_d2 = norm.cdf(d2)
    cdf_nd1 = norm.cdf(-d1)
    cdf_nd2 = norm.cdf(-d2)

    # Delta
    if is_call:
        delta = math.exp(-q * T) * cdf_d1
    else:
        delta = math.exp(-q * T) * (cdf_d1 - 1)

    # Gamma (same for calls and puts)
    gamma = math.exp(-q * T) * pdf_d1 / (S * sigma * sqrtT)

    # Vega
    vega = S * math.exp(-q * T) * pdf_d1 * sqrtT  # per X volatility (not %)

    # Theta (per year)
    if is_call:
        theta = (
            - (S * pdf_d1 * sigma * math.exp(-q * T)) / (2 * sqrtT)
            - r * K * math.exp(-r * T) * cdf_d2
            + q * S * math.exp(-q * T) * cdf_d1
        )
        rho = K * T * math.exp(-r * T) * cdf_d2
    else:
        theta = (
            - (S * pdf_d1 * sigma * math.exp(-q * T)) / (2 * sqrtT)
            + r * K * math.exp(-r * T) * cdf_nd2
            - q * S * math.exp(-q * T) * cdf_nd1
        )
        rho = -K * T * math.exp(-r * T) * cdf_nd2

    # Convert theta from per-year to per-day if desired:
    # theta /= X

    return round(delta, 4), round(gamma, 6), round(vega / 100, 4), round(theta / 365, 4), round(rho / 100, 4)


In [ ]:
def find_spot(img):
    # Get dimensions
    h, w, _ = img.shape

    # Define subregions
    left_section = img[:int(h * 0.2), :int(w * 0.33)]
    top_right_section = img[:int(h * 0.33), int(w * 0.33):]

    # Convert to grayscale and preprocess for OCR
    def preprocess(img):
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
        return gray

    left_pre = preprocess(left_section)

    # OCR for each section
    text_left = pytesseract.image_to_string(left_pre, config='--psm 6')

    # Print raw OCR results
    # print("=== Left Section (Spot, Strike, Breakeven) ===")
    # print(text_left)

    match = re.search(r"\$(\d+(?:\.\d+)?)", text_left)

    if match:
        spot_price = float(match.group(1))
        return spot_price, 'Call' in text_left
    else:
        print("No price found")

In [ ]:
def find_option_chain(img):
    # Keep only near-black text (filter out colored text)
    # Define "dark" pixels as those with low brightness and low color difference
    h, w, _ = img.shape
    left_section = img[:, :int(w * 0.33)]
    data = np.array(left_section)
    gray = np.mean(data, axis=2)
    color_diff = np.max(data, axis=2) - np.min(data, axis=2)
    mask = (gray < 200) & (color_diff < 80)  # dark and low-saturation (near black)

    # Create a white background and overlay black text
    filtered = np.ones_like(data) * 255
    filtered[mask] = 0
    filtered_img = Image.fromarray(filtered.astype('uint8'))

    # Run OCR to extract text
    text = pytesseract.image_to_string(filtered_img, config='--psm 6')
    # print(text)
    # filtered_img.save("filtered_black_text.png")

    text = text.replace(',', ' ')
    pattern = re.compile(r"\$([\d]+(?:\.\d+)?)\s*\$([\d]+(?:\.\d+)?)")
    pairs = []

    for match in pattern.finditer(text):
        strike = float(match.group(1))
        breakeven = float(match.group(2))
        pairs.append((strike, round(breakeven-strike,2)))
    return pairs


In [ ]:
def find_expiry(img, time_capture):
    h, w, _ = img.shape
    left_section = img[:int(h*0.2), int(w * 0.33):int(w * 0.8)]
    data = np.array(left_section)
    gray = np.mean(data, axis=2)
    color_diff = np.max(data, axis=2) - np.min(data, axis=2)
    mask = (gray < 200) & (color_diff < 200) # dark and low-saturation (near black)

    # Create a white background and overlay black text
    filtered = np.ones_like(data) * 255
    filtered[mask] = 0
    filtered_img = Image.fromarray(filtered.astype('uint8'))

    # Run OCR to extract text
    text = pytesseract.image_to_string(filtered_img, config='--psm 6')
    # print(text)
    # filtered_img.save("filtered_black_text.png")
    text = text.replace(',', ' ')

    pattern = re.compile(
        r"Expiring\s*([A-Z][a-z]+)\s*(\d{1,2})(?:\s*(\d{4}))?",
        re.IGNORECASE
    )

    match = pattern.search(text)
    if match:
        month, day, year = match.groups()
        if not year:
            year = "2025"  # fallback default year
        # print(month, day, year)

    match = re.search(r"\((\d+)d\)", text)
    month_num = datetime.strptime(month, "%B").month
    expiry_date = datetime(int(year), month_num, int(day))

    if match:
        days = int(match.group(1))
        return days
    else:
        # print("No day count found")
        # filtered_img.save("filtered_black_text.png")
        return (expiry_date - time_capture).days

In [ ]:
def estimate_screenshot_date(files):

    # Run find_option_chain() on each file
    for f in files:
        img_path = os.path.join(img_dir, f)
        img = cv2.imread(img_path)
        h, w, _ = img.shape
        left_section = img[:int(h*0.2), int(w * 0.33):int(w * 0.8)]
        data = np.array(left_section)
        gray = np.mean(data, axis=2)
        color_diff = np.max(data, axis=2) - np.min(data, axis=2)
        mask = (gray < 200) & (color_diff < 200) # dark and low-saturation (near black)

        # Create a white background and overlay black text
        filtered = np.ones_like(data) * 255
        filtered[mask] = 0
        filtered_img = Image.fromarray(filtered.astype('uint8'))

        # Run OCR to extract text
        text = pytesseract.image_to_string(filtered_img, config='--psm 6')
        # print(text)
        # filtered_img.save("filtered_black_text.png")
        text = text.replace(',', ' ')

        pattern = re.compile(
            r"Expiring\s*([A-Z][a-z]+)\s*(\d{1,2})(?:\s*(\d{4}))?",
            re.IGNORECASE
        )

        match = pattern.search(text)
        if match:
            month, day, year = match.groups()
            if not year:
                year = "2025"  # fallback default year
            # print(month, day, year)

        match = re.search(r"\((\d+)d\)", text)
        month_num = datetime.strptime(month, "%B").month
        expiry_date = datetime(int(year), month_num, int(day))

        if match:
            days = int(match.group(1))
            return expiry_date - timedelta(days)


In [ ]:
def create_full_option_chain(spot, is_call, expiry, prices, r=0.04):
    chain = []
    for strike, price in prices:
        T = expiry / 365
        iv = implied_vol(spot, strike, T, r, price, is_call=is_call)
        delta, gamma, vega, theta, rho = bs_greeks(spot, strike, T, r, 0, iv, is_call)

        contract = [
            float(spot), bool(is_call), float(strike), int(expiry),
            round(float(price), 2), float(r), round(float(iv), 4),
            float(delta), float(gamma), float(vega),
            float(theta), float(rho)
        ]
        chain.append(contract)
    df = pd.DataFrame(chain, columns=[
        "spot", "is_call", "strike", "expiry_days", "price",
        "risk_free_rate", "iv", "delta", "gamma", "vega", "theta", "rho"
    ])
    return df

In [ ]:
img_dir = "images"

files = [f for f in os.listdir(img_dir) if f.startswith("GLDIV") and f.endswith(".png")]
files.sort()  # Optional: process in sorted orde
date_captured = estimate_screenshot_date(files)
print(date_captured)
option_chain = []

for f in files:
    img_path = os.path.join(img_dir, f)
    img = cv2.imread(img_path)
    print(f"Processing {f} ...")
    try:
        spot, is_call = find_spot(img)
        expiry = find_expiry(img, date_captured)
        # print(expiry)
        # print(spot, is_call)
        prices = find_option_chain(img)
        # print(result)
        option_chain.append(create_full_option_chain(spot, is_call, expiry, prices))
    except Exception as e:
        print(f"Error processing {f}: {e}")
        print("-" * 60)
option_chain_df = pd.concat(option_chain, ignore_index=True)
print(option_chain_df)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_filename = f"option_chain_{timestamp}.csv"
csv_path = os.path.join(img_dir, csv_filename)

option_chain_df.to_csv(csv_path, index=False)
print(f"\n✅ Saved option chain to: {csv_path}")